In [1]:
import os
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

In [2]:
X = np.load("X_preprocessed_conf1405.npy")
y_encoded = np.load("y_labels_conf1405.npy")
class_names = ["AMD", "DME", "ERM", "NO", "RAO", "RVO", "VID"]

In [3]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

In [4]:
#Create Test Dataset and Test Loader
batch_size = 16

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [5]:
# Define Stratified 5-Fold Cross Validation
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [6]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    ),
    transforms.RandomAdjustSharpness(sharpness_factor=1.5, p=0.3),
    transforms.RandomAutocontrast(p=0.3)
])

In [7]:
def prepare_for_convnext(images):

    # Convert grayscale 1-channel OCT image to 3-channel
    if images.shape[1] == 1:
        images = images.repeat(1, 3, 1, 1)

    images = images.to(device)

    # ImageNet normalization for pretrained ConvNeXt
    mean = torch.tensor(
        [0.485, 0.456, 0.406],
        dtype=torch.float32,
        device=device
    ).view(1, 3, 1, 1)

    std = torch.tensor(
        [0.229, 0.224, 0.225],
        dtype=torch.float32,
        device=device
    ).view(1, 3, 1, 1)

    images = (images - mean) / std

    return images

In [8]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(
            inputs,
            targets,
            weight=self.alpha,
            reduction="none"
        )

        pt = torch.exp(-ce_loss)

        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        else:
            return focal_loss

In [9]:
# Coordinate Attention

class CoordinateAttention(nn.Module):
    def __init__(self, in_channels, reduction=32):
        super(CoordinateAttention, self).__init__()

        reduced_channels = max(8, in_channels // reduction)

        self.conv1 = nn.Conv2d(in_channels, reduced_channels, kernel_size=1)
        self.bn1 = nn.BatchNorm2d(reduced_channels)
        self.act = nn.Hardswish()

        self.conv_h = nn.Conv2d(reduced_channels, in_channels, kernel_size=1)
        self.conv_w = nn.Conv2d(reduced_channels, in_channels, kernel_size=1)

    def forward(self, x):
        identity = x

        n, c, h, w = x.size()

        x_h = F.adaptive_avg_pool2d(x, (h, 1))
        x_w = F.adaptive_avg_pool2d(x, (1, w))

        x_w = x_w.permute(0, 1, 3, 2)

        y = torch.cat([x_h, x_w], dim=2)

        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)

        x_h, x_w = torch.split(y, [h, w], dim=2)

        x_w = x_w.permute(0, 1, 3, 2)

        a_h = torch.sigmoid(self.conv_h(x_h))
        a_w = torch.sigmoid(self.conv_w(x_w))

        return identity * a_h * a_w

In [10]:
# ConvNeXt-Tiny + Coordinate Attention

class ConvNeXtTinyWithCoordinateAttention(nn.Module):
    def __init__(self, num_classes):
        super(ConvNeXtTinyWithCoordinateAttention, self).__init__()

        weights = ConvNeXt_Tiny_Weights.DEFAULT
        self.model = convnext_tiny(weights=weights)

        self.coord_attention = CoordinateAttention(in_channels=768)

        in_features = self.model.classifier[2].in_features
        self.model.classifier[2] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        x = self.model.features(x)
        x = self.coord_attention(x)
        x = self.model.avgpool(x)
        x = self.model.classifier(x)

        return x

In [11]:
# Dataset class for augmentation

class OCTDataset(torch.utils.data.Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = torch.tensor(self.images[idx], dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label

In [12]:
# Create Train and Validation DataLoaders for Each Fold
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

num_classes = 7
num_epochs = 50
batch_size = 16

fold_results = []

for fold, (train_index, val_index) in enumerate(skf.split(X_trainval, y_trainval), 1):

    print("\n" + "=" * 60)
    print(f"Starting Fold {fold}")
    print("=" * 60)

    X_train = X_trainval[train_index]
    X_val = X_trainval[val_index]

    y_train = y_trainval[train_index]
    y_val = y_trainval[val_index]

    print("Train shape:", X_train.shape, y_train.shape)
    print("Val shape:", X_val.shape, y_val.shape)

    train_dataset = OCTDataset(X_train, y_train, transform=train_transform)
    val_dataset   = OCTDataset(X_val, y_val, transform=None)

    # Count samples in each class for this fold
    class_sample_count = np.array([
        len(np.where(y_train == t)[0]) for t in np.unique(y_train)
    ])
    
    # Weight for each class
    class_weights_sampler = 1.0 / class_sample_count
    
    # Assign weight to each training sample
    samples_weight = np.array([class_weights_sampler[label] for label in y_train])
    samples_weight = torch.from_numpy(samples_weight).double()
    
    # Create sampler
    sampler = WeightedRandomSampler(
        weights=samples_weight,
        num_samples=len(samples_weight),
        replacement=True
    )
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=sampler
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    # Class weights for imbalance handling
    classes = np.unique(y_train)

    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )

    class_weights = torch.tensor(
        class_weights,
        dtype=torch.float32
    ).to(device)

    print("Class weights:", class_weights)

    # Fresh model for each fold
    model = ConvNeXtTinyWithCoordinateAttention(num_classes=num_classes)
    model = model.to(device)

    # Freeze all feature layers first
    for param in model.model.features.parameters():
        param.requires_grad = False
    
    # Unfreeze last 2 ConvNeXt feature blocks
    for param in model.model.features[-2:].parameters():
        param.requires_grad = True
    
    # Coordinate Attention trainable
    for param in model.coord_attention.parameters():
        param.requires_grad = True
    
    # Classifier trainable
    for param in model.model.classifier.parameters():
        param.requires_grad = True

    criterion = FocalLoss(
        alpha=class_weights,
        gamma=2.0
    )

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-4,
        weight_decay=1e-4
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=3
    )

    latest_checkpoint_path = f"coordatt_fold1_{fold}_latest2.pth"
    best_checkpoint_path = f"coordatt_fold1_{fold}_best2.pth"

    start_epoch = 0
    best_val_acc = 0.0

    if os.path.exists(latest_checkpoint_path):
        checkpoint = torch.load(latest_checkpoint_path, map_location=device)

        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

        start_epoch = checkpoint["epoch"] + 1
        best_val_acc = checkpoint["best_val_acc"]

        print(f"Resuming Fold {fold} from Epoch {start_epoch + 1}")
    else:
        print(f"Starting Fold {fold} from Epoch 1")

    for epoch in range(start_epoch, num_epochs):

        model.train()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        train_bar = tqdm(
            train_loader,
            desc=f"Fold {fold}/5 | Epoch {epoch+1}/{num_epochs} [Train]"
        )

        for images, labels in train_bar:

            images = prepare_for_convnext(images)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

            running_train_acc = 100 * train_correct / train_total

            train_bar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "acc": f"{running_train_acc:.2f}%"
            })

        train_loss = train_loss / len(train_loader)
        train_acc = 100 * train_correct / train_total

        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        val_bar = tqdm(
            val_loader,
            desc=f"Fold {fold}/5 | Epoch {epoch+1}/{num_epochs} [Val]"
        )

        with torch.no_grad():
            for images, labels in val_bar:

                images = prepare_for_convnext(images)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)

                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

                running_val_acc = 100 * val_correct / val_total

                val_bar.set_postfix({
                    "loss": f"{loss.item():.4f}",
                    "acc": f"{running_val_acc:.2f}%"
                })

        val_loss = val_loss / len(val_loader)
        val_acc = 100 * val_correct / val_total
        scheduler.step(val_acc)

        print(
            f"Fold [{fold}/5] "
            f"Epoch [{epoch+1}/{num_epochs}] "
            f"Train Loss: {train_loss:.4f} "
            f"Train Acc: {train_acc:.2f}% "
            f"Val Loss: {val_loss:.4f} "
            f"Val Acc: {val_acc:.2f}%"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc

            torch.save({
                "fold": fold,
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_acc": best_val_acc
            }, best_checkpoint_path)

            print(f"Best checkpoint saved for Fold {fold}")

        torch.save({
            "fold": fold,
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_acc": best_val_acc
        }, latest_checkpoint_path)

        print(f"Latest checkpoint saved for Fold {fold}")

    print(f"Fold {fold} completed. Best Val Acc: {best_val_acc:.2f}%")

    fold_results.append(best_val_acc)


print("\n" + "=" * 60)
print("5-Fold Cross Validation Results")
print("=" * 60)

for i, acc in enumerate(fold_results, 1):
    print(f"Fold {i} Best Val Accuracy: {acc:.2f}%")

mean_acc = np.mean(fold_results)
std_acc = np.std(fold_results)

print(f"\nMean Validation Accuracy: {mean_acc:.2f}%")
print(f"Standard Deviation: {std_acc:.2f}%")


Using device: cpu

Starting Fold 1
Train shape: (1035, 1, 224, 224) (1035,)
Val shape: (259, 1, 224, 224) (259,)
Class weights: tensor([0.2612, 1.6071, 1.7602, 0.8169, 9.8571, 2.5061, 3.8910])
Resuming Fold 1 from Epoch 51
Fold 1 completed. Best Val Acc: 78.76%

Starting Fold 2
Train shape: (1035, 1, 224, 224) (1035,)
Val shape: (259, 1, 224, 224) (259,)
Class weights: tensor([ 0.2612,  1.6248,  1.7395,  0.8169, 10.5612,  2.4643,  3.8910])
Resuming Fold 2 from Epoch 51
Fold 2 completed. Best Val Acc: 82.63%

Starting Fold 3
Train shape: (1035, 1, 224, 224) (1035,)
Val shape: (259, 1, 224, 224) (259,)
Class weights: tensor([ 0.2612,  1.6248,  1.7395,  0.8124, 10.5612,  2.5061,  3.8910])
Resuming Fold 3 from Epoch 47


Fold 3/5 | Epoch 47/50 [Train]: 100%|█████████████████████████| 65/65 [02:36<00:00,  2.41s/it, loss=0.0034, acc=95.75%]
Fold 3/5 | Epoch 47/50 [Val]: 100%|███████████████████████████| 17/17 [00:19<00:00,  1.15s/it, loss=0.0312, acc=85.33%]


Fold [3/5] Epoch [47/50] Train Loss: 0.0226 Train Acc: 95.75% Val Loss: 0.2903 Val Acc: 85.33%
Latest checkpoint saved for Fold 3


Fold 3/5 | Epoch 48/50 [Train]: 100%|█████████████████████████| 65/65 [02:20<00:00,  2.16s/it, loss=0.0487, acc=94.78%]
Fold 3/5 | Epoch 48/50 [Val]: 100%|███████████████████████████| 17/17 [00:16<00:00,  1.03it/s, loss=0.0982, acc=83.40%]


Fold [3/5] Epoch [48/50] Train Loss: 0.0287 Train Acc: 94.78% Val Loss: 0.2977 Val Acc: 83.40%
Latest checkpoint saved for Fold 3


Fold 3/5 | Epoch 49/50 [Train]: 100%|█████████████████████████| 65/65 [01:37<00:00,  1.50s/it, loss=0.0177, acc=95.65%]
Fold 3/5 | Epoch 49/50 [Val]: 100%|███████████████████████████| 17/17 [00:16<00:00,  1.01it/s, loss=0.0231, acc=86.10%]


... (79 outputs left out)

Fold 5/5 | Epoch 49/50 [Train]: 100%|█████████████████████████| 65/65 [02:21<00:00,  2.17s/it, loss=0.0045, acc=95.37%]
Fold 5/5 | Epoch 49/50 [Val]: 100%|███████████████████████████| 17/17 [00:25<00:00,  1.48s/it, loss=0.0004, acc=86.05%]


Fold [5/5] Epoch [49/50] Train Loss: 0.0172 Train Acc: 95.37% Val Loss: 0.5279 Val Acc: 86.05%
Best checkpoint saved for Fold 5
Latest checkpoint saved for Fold 5


Fold 5/5 | Epoch 50/50 [Train]: 100%|█████████████████████████| 65/65 [02:24<00:00,  2.22s/it, loss=0.0025, acc=96.43%]
Fold 5/5 | Epoch 50/50 [Val]: 100%|███████████████████████████| 17/17 [00:25<00:00,  1.50s/it, loss=0.0003, acc=84.11%]


Fold [5/5] Epoch [50/50] Train Loss: 0.0179 Train Acc: 96.43% Val Loss: 0.5268 Val Acc: 84.11%
Latest checkpoint saved for Fold 5
Fold 5 completed. Best Val Acc: 86.05%

5-Fold Cross Validation Results
Fold 1 Best Val Accuracy: 78.76%
Fold 2 Best Val Accuracy: 82.63%
Fold 3 Best Val Accuracy: 86.10%
Fold 4 Best Val Accuracy: 82.63%
Fold 5 Best Val Accuracy: 86.05%

Mean Validation Accuracy: 83.23%
Standard Deviation: 2.71%


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = len(np.unique(y_encoded))

checkpoint_paths = [
    "coordatt_fold1_1_best2.pth",
    "coordatt_fold1_2_best2.pth",
    "coordatt_fold1_3_best2.pth",
    "coordatt_fold1_4_best2.pth",
    "coordatt_fold1_5_best2.pth"
]

models = []

for path in checkpoint_paths:

    model = ConvNeXtTinyWithCoordinateAttention(num_classes=num_classes)

    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])

    model = model.to(device)
    model.eval()

    models.append(model)

print("All 5 Coordinate Attention fold models loaded successfully")

All 5 Coordinate Attention fold models loaded successfully


In [14]:
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader):

        images = prepare_for_convnext(images)
        images = images.to(device)
        labels = labels.to(device)

        fold_probs = []

        for model in models:
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            fold_probs.append(probs)

        # Average probabilities from 5 fold models
        avg_probs = torch.mean(torch.stack(fold_probs), dim=0)

        # Final prediction
        _, predicted = torch.max(avg_probs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

100%|██████████████████████████████████████████████████████████████████████████████████| 21/21 [01:56<00:00,  5.55s/it]


In [23]:
accuracy = accuracy_score(all_labels, all_preds)

print("Ensemble Test Accuracy:", accuracy * 100)

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

Ensemble Test Accuracy: 79.93827160493827

Classification Report:
              precision    recall  f1-score   support

         AMD       1.00      0.72      0.84       177
         DME       0.63      0.76      0.69        29
         ERM       0.85      0.85      0.85        27
          NO       0.61      0.98      0.75        57
         RAO       0.60      0.75      0.67         4
         RVO       0.77      0.89      0.83        19
         VID       0.69      1.00      0.81        11

    accuracy                           0.80       324
   macro avg       0.74      0.85      0.78       324
weighted avg       0.86      0.80      0.81       324


Confusion Matrix:
[[127  12   2  27   1   3   5]
 [  0  22   2   3   0   2   0]
 [  0   0  23   4   0   0   0]
 [  0   0   0  56   1   0   0]
 [  0   0   0   1   3   0   0]
 [  0   1   0   1   0  17   0]
 [  0   0   0   0   0   0  11]]


In [24]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(
    all_labels,
    all_preds,
    target_names=class_names,
    digits=4,
    zero_division=0
))

print(confusion_matrix(all_labels, all_preds))

              precision    recall  f1-score   support

         AMD     1.0000    0.7175    0.8355       177
         DME     0.6286    0.7586    0.6875        29
         ERM     0.8519    0.8519    0.8519        27
          NO     0.6087    0.9825    0.7517        57
         RAO     0.6000    0.7500    0.6667         4
         RVO     0.7727    0.8947    0.8293        19
         VID     0.6875    1.0000    0.8148        11

    accuracy                         0.7994       324
   macro avg     0.7356    0.8507    0.7768       324
weighted avg     0.8567    0.7994    0.8057       324

[[127  12   2  27   1   3   5]
 [  0  22   2   3   0   2   0]
 [  0   0  23   4   0   0   0]
 [  0   0   0  56   1   0   0]
 [  0   0   0   1   3   0   0]
 [  0   1   0   1   0  17   0]
 [  0   0   0   0   0   0  11]]
